In [8]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import re

# --- Configuration ---
TELEMETRY_BASE_DIR = 'Telemetry/Real'
SECTIONS_BASE_DIR = 'Sections/Real'
PLOTS_OUTPUT_DIR = 'Plots/Real'

Y_AXIS_COLUMNS = [
    'Time', 'car_speed', 'STEER', 'P_F_BRAKE', 'PPS.proportion',
    'flDamper_Corr', 'frDamper_Corr', 'rlDamper_Corr', 'rrDamper_Corr'
]

# The first 3 colors will be used for laps
LAP_COLORS = [
    'blue', 'red', 'green', 'orange', 'purple',
    'cyan', 'magenta', 'brown', 'pink', 'gray'
]

# --- Main Script ---
def create_plots():
    """
    Generates telemetry plots with highly visible vertical section lines.
    """
    # --- 1. Set up directories ---
    if not os.path.exists(PLOTS_OUTPUT_DIR):
        print(f"Creating output directory: {PLOTS_OUTPUT_DIR}")
        os.makedirs(PLOTS_OUTPUT_DIR)

    overwrite_all = None
    while overwrite_all is None:
        response = input("Do you want to overwrite existing plot files? (yes/no): ").lower()
        if response in ['yes', 'y']:
            overwrite_all = True
        elif response in ['no', 'n']:
            overwrite_all = False
        else:
            print("Invalid input. Please enter 'yes' or 'no'.")
    
    # --- 2. Iterate through each track folder ---
    if not os.path.isdir(TELEMETRY_BASE_DIR):
        print(f"Error: Telemetry directory not found at '{TELEMETRY_BASE_DIR}'")
        return

    track_names = [d for d in os.listdir(TELEMETRY_BASE_DIR) if os.path.isdir(os.path.join(TELEMETRY_BASE_DIR, d))]

    for track_name in track_names:
        if track_name.startswith('Target'):
            print(f"\nSkipping target directory: {track_name}")
            continue

        track_path = os.path.join(TELEMETRY_BASE_DIR, track_name)
        print(f"\nProcessing track: {track_name}")

        output_filename = os.path.join(PLOTS_OUTPUT_DIR, f"plot_telemetry_{track_name}.png")

        if not overwrite_all and os.path.exists(output_filename):
            print(f"  Skipping '{track_name}' as plot already exists.")
            continue

        try:
            telemetry_files = sorted([f for f in os.listdir(track_path) if f.startswith('lap') and f.endswith('.csv')])
            if not telemetry_files:
                print(f"  No telemetry files found for track '{track_name}'. Skipping.")
                continue
        except FileNotFoundError:
            print(f"  Directory not found: {track_path}. Skipping.")
            continue

        # --- 3. Pre-load all lap data for the track ---
        lap_data_list = []
        for file_name in telemetry_files:
            file_path = os.path.join(track_path, file_name)
            try:
                lap_match = re.search(r'lap(\d+)', file_name)
                lap_label = f"Lap {lap_match.group(1)}" if lap_match else file_name
                df = pd.read_csv(file_path)
                lap_data_list.append({'df': df, 'label': lap_label})
            except Exception as e:
                print(f"  Error processing telemetry file '{file_name}': {e}. Skipping this file.")
        
        if not lap_data_list:
            print(f"  No telemetry data could be loaded for track '{track_name}'. Skipping plot generation.")
            continue

        # --- 4. Load section data for the track ---
        section_distances = []
        section_file_path = os.path.join(SECTIONS_BASE_DIR, f"sections_{track_name}.csv")
        if os.path.exists(section_file_path):
            try:
                sections_df = pd.read_csv(section_file_path)
                if 'start_m' in sections_df.columns:
                    section_distances = sections_df['start_m'].tolist()
                    print(f"  Loaded {len(section_distances)} section starts from {section_file_path}")
                else:
                    print(f"  Warning: 'start_m' column not found in {section_file_path}. No section lines will be drawn.")
            except Exception as e:
                print(f"  Error reading section file {section_file_path}: {e}")
        else:
            print(f"  Info: No section file found at '{section_file_path}'.")


        # --- 5. Create the plot with vertical subplots ---
        num_plots = len(Y_AXIS_COLUMNS)
        fig, axs = plt.subplots(nrows=num_plots, ncols=1, figsize=(15, 3 * num_plots), sharex=True)
        if num_plots == 1: axs = [axs]

        for plot_index, column in enumerate(Y_AXIS_COLUMNS):
            current_ax = axs[plot_index]
            for lap_index, lap_data in enumerate(lap_data_list):
                lap_color = LAP_COLORS[lap_index % len(LAP_COLORS)]
                df = lap_data['df']
                if column in df.columns:
                    current_ax.plot(df['Distance'], df[column], color=lap_color)
            current_ax.set_ylabel(column)
            current_ax.grid(True)

        # --- 6. Draw vertical section lines on all subplots ---
        if section_distances:
            for ax in axs:
                for dist in section_distances:
                    ### CHANGED: Made the line thicker, solid, and a more visible color ###
                    ax.axvline(x=dist, color='purple', linestyle='-', linewidth=2.5, zorder=5)

        # --- 7. Finalise and save the plot ---
        legend_handles = []
        for i, lap_data in enumerate(lap_data_list):
            legend_handles.append(plt.Line2D([0], [0], color=LAP_COLORS[i % len(LAP_COLORS)], lw=4, label=lap_data['label']))
        
        fig.legend(handles=legend_handles, title='Laps', bbox_to_anchor=(1.01, 0.95), loc='upper left')

        formatted_track_name = track_name.replace("_", " ").title()
        fig.suptitle(f"Telemetry Data for {formatted_track_name}", fontsize=16)

        axs[-1].set_xlabel('Distance (meters)')
        
        plt.tight_layout(rect=[0, 0, 1, 0.96])

        print(f"  Saving plot to {output_filename}")
        plt.savefig(output_filename, dpi=300, bbox_inches='tight')
        plt.close(fig)

    print("\nScript finished.")

if __name__ == '__main__':
    create_plots()


Processing track: silverstone_gp
  Loaded 7 section starts from Sections/Real\sections_silverstone_gp.csv
  Saving plot to Plots/Real\plot_telemetry_silverstone_gp.png

Processing track: brands_gp
  Skipping 'brands_gp' as plot already exists.

Processing track: hungaroring
  Skipping 'hungaroring' as plot already exists.

Processing track: donington_gp
  Skipping 'donington_gp' as plot already exists.

Processing track: spa
  Skipping 'spa' as plot already exists.

Processing track: zandvoort
  Skipping 'zandvoort' as plot already exists.

Skipping target directory: Target_Monza

Script finished.
